# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_15 — Purged Feature Selection for Time Series

### Research question

How can feature selection for financial time-series models be performed without leaking information through overlapping labels, future-derived features, or random cross-validation?

This notebook follows naturally from:

```text
03_14_information_coefficient_and_feature_decay
03_11_factor_orthogonalization_and_eviction
03_07_lstm_forecaster
```

The previous notebook measured feature health. This notebook asks a different question:

> If labels overlap through time, how do we select features without letting training samples contain information about validation samples?

It covers:

1. overlapping forward-return labels;
2. why random K-fold cross-validation leaks in finance;
3. blocked time-series folds;
4. purged cross-validation;
5. embargo after validation windows;
6. feature availability audits;
7. intentionally leaky feature demonstration;
8. filter-based feature selection inside each fold;
9. ridge model benchmark;
10. naive CV versus purged CV results;
11. feature selection stability;
12. final train/test evaluation;
13. leakage checklist;
14. saved reports and manifest.

The key idea:

> In financial ML, feature selection must be inside a leakage-safe validation loop. Otherwise the selected features can be artifacts of overlapping labels, future information, or random split contamination.

## 1. Why ordinary cross-validation fails

Suppose the label is a 21-day forward return:

$$
y_t = \log(P_{t+21}/P_t)
$$

Then label $y_t$ uses returns from:

$$
(t+1,\dots,t+21)
$$

and label $y_{t+1}$ uses:

$$
(t+2,\dots,t+22)
$$

These labels overlap heavily.

If a random split puts $y_t$ in train and $y_{t+1}$ in validation, the training label and validation label share almost all underlying future returns.

That can make validation performance look better than it really is.

This is why time-series feature selection needs purging.

## 2. Event intervals

Each sample is an event:

$$
i = (t_{0,i}, t_{1,i})
$$

where:

- $t_0$: time the feature is observed;
- $t_1$: time the label finishes.

For a forward $H$-day return:

$$
t_1 = t_0 + H
$$

Two samples overlap if their intervals intersect:

$$
[t_{0,i}, t_{1,i}] \cap [t_{0,j}, t_{1,j}] \neq \emptyset
$$

If validation events cover an interval, training events whose labels overlap that interval should be removed.

## 3. Purging and embargo

### Purging

For a validation interval:

$$
[V_0,V_1]
$$

remove any training event $i$ such that:

$$
[t_{0,i},t_{1,i}] \cap [V_0,V_1] \neq \emptyset
$$

### Embargo

After the validation interval, remove observations in a small buffer:

$$
(V_1, V_1+\delta]
$$

This reduces leakage from serial dependence, delayed effects, and feature construction windows.

Important:

> Purging and embargo help with overlapping labels. They do not fix features that directly use future data. Feature availability must be audited separately.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class PurgedFeatureSelectionConfig:
    n_obs: int = 2_500
    label_horizon: int = 21
    train_fraction: float = 0.70
    n_splits: int = 5
    embargo_pct: float = 0.02
    ridge_lambda: float = 1e-3
    k_grid: tuple = (1, 2, 3, 5, 8, 12, 16)
    seed: int = 42


config = PurgedFeatureSelectionConfig()
config

## 5. Synthetic time-series data

We simulate a noisy market with:

1. latent slow signal;
2. mean-reversion signal;
3. volatility regime;
4. noisy candidate features;
5. redundant feature clones;
6. one intentionally leaky feature.

The leaky feature uses future information and should be caught by an availability audit.

This demonstrates two distinct problems:

1. **feature look-ahead leakage** — fixed by feature audit;
2. **label overlap leakage** — fixed by purging and embargo.

In [ ]:
def simulate_time_series_features(config: PurgedFeatureSelectionConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(config.seed)
    n = config.n_obs
    H = config.label_horizon

    dates = pd.bdate_range("2015-01-01", periods=n)

    latent_trend = np.zeros(n)
    latent_reversal = np.zeros(n)
    latent_vol = np.zeros(n)
    returns = np.zeros(n)

    latent_vol[0] = 0.012

    for t in range(2, n):
        latent_trend[t] = 0.97 * latent_trend[t - 1] + 0.10 * rng.standard_normal()
        latent_reversal[t] = -0.40 * returns[t - 1] + 0.15 * rng.standard_normal()
        latent_vol[t] = 0.94 * latent_vol[t - 1] + 0.06 * (0.008 + 0.012 * (rng.uniform() < 0.05)) + 0.05 * abs(returns[t - 1])
        latent_vol[t] = np.clip(latent_vol[t], 0.003, 0.060)

        weak_mean = 0.00005 + 0.00035 * latent_trend[t - 1] + 0.00025 * latent_reversal[t - 1]
        shock = latent_vol[t] * rng.standard_t(df=7) * np.sqrt((7 - 2) / 7)

        returns[t] = weak_mean + shock

    price = 100 * np.exp(np.cumsum(returns))

    df = pd.DataFrame({
        "date": dates,
        "return": returns,
        "price": price,
        "latent_trend": latent_trend,
        "latent_reversal": latent_reversal,
        "latent_vol": latent_vol
    })

    # Backward-looking features.
    df["momentum_5"] = np.log(df["price"] / df["price"].shift(5))
    df["momentum_21"] = np.log(df["price"] / df["price"].shift(21))
    df["reversal_1"] = -df["return"]
    df["rolling_vol_21"] = df["return"].rolling(21).std()
    df["rolling_vol_63"] = df["return"].rolling(63).std()
    df["vol_ratio_21_63"] = df["rolling_vol_21"] / df["rolling_vol_63"]
    df["drawdown_63"] = df["price"] / df["price"].rolling(63).max() - 1
    df["trend_proxy"] = df["latent_trend"] + 0.30 * rng.standard_normal(n)
    df["reversal_proxy"] = df["latent_reversal"] + 0.30 * rng.standard_normal(n)

    # Redundant features.
    df["momentum_clone"] = 0.92 * df["momentum_21"] + 0.02 * rng.standard_normal(n)
    df["vol_clone"] = 0.90 * df["rolling_vol_21"] + 0.002 * rng.standard_normal(n)

    # Noise features.
    for i in range(1, 11):
        df[f"noise_{i}"] = rng.standard_normal(n)

    # Labels.
    df["forward_return_H"] = np.log(df["price"].shift(-H) / df["price"])
    df["forward_return_1d"] = df["return"].shift(-1)

    # Intentionally leaky feature: uses the forward label plus noise.
    df["LEAK_future_return_hint"] = df["forward_return_H"] + 0.02 * rng.standard_normal(n)

    df["t0"] = df["date"]
    df["t1"] = df["date"].shift(-H)

    # Drop rows without feature history or label future.
    df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    feature_metadata = pd.DataFrame([
        {"feature": "momentum_5", "uses_future": False, "description": "5-day trailing momentum"},
        {"feature": "momentum_21", "uses_future": False, "description": "21-day trailing momentum"},
        {"feature": "reversal_1", "uses_future": False, "description": "negative current return"},
        {"feature": "rolling_vol_21", "uses_future": False, "description": "21-day trailing volatility"},
        {"feature": "rolling_vol_63", "uses_future": False, "description": "63-day trailing volatility"},
        {"feature": "vol_ratio_21_63", "uses_future": False, "description": "trailing vol ratio"},
        {"feature": "drawdown_63", "uses_future": False, "description": "trailing drawdown"},
        {"feature": "trend_proxy", "uses_future": False, "description": "noisy latent trend proxy"},
        {"feature": "reversal_proxy", "uses_future": False, "description": "noisy latent reversal proxy"},
        {"feature": "momentum_clone", "uses_future": False, "description": "redundant momentum clone"},
        {"feature": "vol_clone", "uses_future": False, "description": "redundant volatility clone"},
        {"feature": "LEAK_future_return_hint", "uses_future": True, "description": "intentional future-label leak"},
    ] + [
        {"feature": f"noise_{i}", "uses_future": False, "description": "pure noise feature"}
        for i in range(1, 11)
    ])

    return df, feature_metadata


events, feature_metadata = simulate_time_series_features(config)

events.head(), feature_metadata.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(events["date"], events["price"])
plt.title("Synthetic Price Series")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(events["date"], events["forward_return_H"])
plt.axhline(0, linestyle="--")
plt.title(f"{config.label_horizon}-Day Forward Return Label")
plt.xlabel("Date")
plt.ylabel("Forward return")
plt.show()

## 6. Feature availability audit

Purging does not solve features that directly use future information.

We first audit features and remove anything marked as future-derived.

In a real system, this audit should be part of the feature store:

- feature timestamp;
- data publication timestamp;
- vendor delay;
- transformation window;
- target horizon;
- asset/event availability.

In [ ]:
all_feature_cols = feature_metadata["feature"].tolist()
leaky_features = feature_metadata.loc[feature_metadata["uses_future"], "feature"].tolist()
safe_feature_cols = feature_metadata.loc[~feature_metadata["uses_future"], "feature"].tolist()

leakage_audit = feature_metadata.copy()
leakage_audit["allowed_for_model"] = ~leakage_audit["uses_future"]

pd.Series({
    "n_all_features": len(all_feature_cols),
    "n_safe_features": len(safe_feature_cols),
    "n_leaky_features_removed": len(leaky_features),
    "leaky_features": ", ".join(leaky_features)
})

In [ ]:
leakage_audit

## 7. Chronological train/test split

The final test set is kept untouched.

Feature selection and model selection happen only inside the research period.

This prevents selecting features based on final test performance.

In [ ]:
split_idx = int(len(events) * config.train_fraction)

research = events.iloc[:split_idx].copy()
test = events.iloc[split_idx:].copy()

split_metadata = {
    "n_total": int(len(events)),
    "n_research": int(len(research)),
    "n_test": int(len(test)),
    "research_start": str(research["date"].iloc[0]),
    "research_end": str(research["date"].iloc[-1]),
    "test_start": str(test["date"].iloc[0]),
    "test_end": str(test["date"].iloc[-1])
}

pd.Series(split_metadata)

## 8. Scaling within train folds

Feature selection should not scale using validation data.

For every fold:

1. fit mean/std on training fold;
2. transform training fold;
3. transform validation fold using training stats.

This avoids leakage from validation distribution into training.

In [ ]:
def fit_scaler(df, feature_cols):
    mean = df[feature_cols].mean()
    std = df[feature_cols].std(ddof=1).replace(0, 1.0)
    return mean, std


def transform_with_scaler(df, feature_cols, mean, std):
    return ((df[feature_cols] - mean) / std).replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy()


def scale_train_valid(train_df, valid_df, feature_cols):
    mean, std = fit_scaler(train_df, feature_cols)
    X_train = transform_with_scaler(train_df, feature_cols, mean, std)
    X_valid = transform_with_scaler(valid_df, feature_cols, mean, std)
    return X_train, X_valid, mean, std

## 9. Ridge regression utilities

We use a simple ridge model:

$$
\hat\beta=(X^\top X+\lambda I)^{-1}X^\top y
$$

This keeps the notebook dependency-light and focuses on validation design rather than model glamour.

In [ ]:
def fit_ridge(X, y, ridge_lambda=1e-3):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    X_design = np.column_stack([np.ones(len(X)), X])
    penalty = ridge_lambda * np.eye(X_design.shape[1])
    penalty[0, 0] = 0.0

    beta = np.linalg.solve(X_design.T @ X_design + penalty, X_design.T @ y)

    return beta


def predict_ridge(beta, X):
    X_design = np.column_stack([np.ones(len(X)), X])
    return X_design @ beta


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    err = y_pred - y_true

    sign_strategy = np.sign(y_pred) * y_true

    return {
        "mse": float(np.mean(err**2)),
        "mae": float(np.mean(np.abs(err))),
        "corr": float(np.corrcoef(y_pred, y_true)[0, 1]) if np.std(y_pred) > 0 and np.std(y_true) > 0 else np.nan,
        "directional_accuracy": float(np.mean(np.sign(y_pred) == np.sign(y_true))),
        "toy_strategy_mean": float(np.mean(sign_strategy)),
        "toy_strategy_vol": float(np.std(sign_strategy, ddof=1)),
        "toy_strategy_ir_daily": float(np.mean(sign_strategy) / np.std(sign_strategy, ddof=1)) if np.std(sign_strategy, ddof=1) > 0 else np.nan
    }

## 10. Fold definitions

We build three splitters:

### Random K-fold

Randomly assigns samples to folds.  
This is **not appropriate** for overlapping financial labels.

### Blocked K-fold

Uses contiguous validation blocks but does not purge overlapping training labels.

### Purged K-fold with embargo

Uses contiguous validation blocks, removes overlapping training events, and applies an embargo after validation.

In [ ]:
def make_random_folds(df, n_splits, seed=42):
    rng = np.random.default_rng(seed)
    indices = np.arange(len(df))
    rng.shuffle(indices)

    folds = np.array_split(indices, n_splits)
    output = []

    for k, valid_idx in enumerate(folds):
        train_idx = np.setdiff1d(indices, valid_idx)
        output.append({
            "fold": k,
            "train_idx": np.sort(train_idx),
            "valid_idx": np.sort(valid_idx),
            "split_type": "random_kfold"
        })

    return output


def make_blocked_folds(df, n_splits):
    indices = np.arange(len(df))
    folds = np.array_split(indices, n_splits)
    output = []

    for k, valid_idx in enumerate(folds):
        train_idx = np.setdiff1d(indices, valid_idx)
        output.append({
            "fold": k,
            "train_idx": np.sort(train_idx),
            "valid_idx": np.sort(valid_idx),
            "split_type": "blocked_kfold_no_purge"
        })

    return output


def intervals_overlap(start_a, end_a, start_b, end_b):
    return (start_a <= end_b) & (end_a >= start_b)


def make_purged_embargo_folds(df, n_splits, embargo_pct):
    indices = np.arange(len(df))
    folds = np.array_split(indices, n_splits)

    t0 = pd.to_datetime(df["t0"]).reset_index(drop=True)
    t1 = pd.to_datetime(df["t1"]).reset_index(drop=True)

    n = len(df)
    embargo_n = int(np.ceil(embargo_pct * n))

    output = []

    for k, valid_idx in enumerate(folds):
        valid_idx = np.asarray(valid_idx)
        valid_start = t0.iloc[valid_idx].min()
        valid_end = t1.iloc[valid_idx].max()

        candidate_train = np.setdiff1d(indices, valid_idx)

        train_t0 = t0.iloc[candidate_train]
        train_t1 = t1.iloc[candidate_train]

        # Purge training samples whose event windows overlap validation event window.
        overlap = intervals_overlap(train_t0, train_t1, valid_start, valid_end)

        # Embargo samples immediately after validation block by index.
        embargo_start_idx = valid_idx.max() + 1
        embargo_end_idx = min(n, embargo_start_idx + embargo_n)
        embargo_idx = np.arange(embargo_start_idx, embargo_end_idx)

        purge_idx = candidate_train[np.asarray(overlap)]
        remove_idx = np.union1d(purge_idx, embargo_idx)

        train_idx = np.setdiff1d(candidate_train, remove_idx)

        output.append({
            "fold": k,
            "train_idx": np.sort(train_idx),
            "valid_idx": np.sort(valid_idx),
            "purged_idx": np.sort(purge_idx),
            "embargo_idx": np.sort(embargo_idx),
            "valid_start": valid_start,
            "valid_end": valid_end,
            "split_type": "purged_embargo_kfold"
        })

    return output


random_folds = make_random_folds(research, config.n_splits, seed=config.seed)
blocked_folds = make_blocked_folds(research, config.n_splits)
purged_folds = make_purged_embargo_folds(research, config.n_splits, config.embargo_pct)

pd.DataFrame([
    {
        "split_type": f["split_type"],
        "fold": f["fold"],
        "n_train": len(f["train_idx"]),
        "n_valid": len(f["valid_idx"]),
        "n_purged": len(f.get("purged_idx", [])),
        "n_embargo": len(f.get("embargo_idx", []))
    }
    for f in purged_folds
])

## 11. Visualising purging and embargo

The purged splitter removes observations whose label windows overlap validation.

This is the central anti-leakage mechanism for overlapping labels.

In [ ]:
fold_diagnostics = []

for fold_set_name, folds in [
    ("random", random_folds),
    ("blocked", blocked_folds),
    ("purged_embargo", purged_folds)
]:
    for f in folds:
        fold_diagnostics.append({
            "fold_set": fold_set_name,
            "fold": f["fold"],
            "n_train": len(f["train_idx"]),
            "n_valid": len(f["valid_idx"]),
            "n_purged": len(f.get("purged_idx", [])),
            "n_embargo": len(f.get("embargo_idx", []))
        })

fold_diagnostics = pd.DataFrame(fold_diagnostics)

fold_diagnostics

In [ ]:
plt.figure(figsize=(10, 6))
for fold_set, g in fold_diagnostics.groupby("fold_set"):
    plt.plot(g["fold"], g["n_train"], marker="o", label=f"{fold_set} train size")
plt.title("Training Sample Count by Fold")
plt.xlabel("Fold")
plt.ylabel("Training samples")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
g = fold_diagnostics[fold_diagnostics["fold_set"] == "purged_embargo"]
plt.bar(g["fold"], g["n_purged"], label="purged")
plt.bar(g["fold"], g["n_embargo"], bottom=g["n_purged"], label="embargo")
plt.title("Purged and Embargoed Samples by Fold")
plt.xlabel("Fold")
plt.ylabel("Samples removed")
plt.legend()
plt.show()

## 12. Feature selection inside each fold

For each fold:

1. use only training fold to rank features;
2. select top $k$ by absolute train correlation with target;
3. fit ridge model on selected features;
4. predict validation fold;
5. record performance.

This avoids selecting features using validation data.

In [ ]:
def rank_features_by_train_correlation(train_df, feature_cols, target_col):
    rows = []

    y = train_df[target_col]

    for feature in feature_cols:
        corr = train_df[feature].corr(y)
        rows.append({
            "feature": feature,
            "train_corr": corr,
            "abs_train_corr": abs(corr)
        })

    return pd.DataFrame(rows).sort_values("abs_train_corr", ascending=False)


def run_cv_feature_selection(df, feature_cols, folds, k_grid, target_col="forward_return_H", ridge_lambda=1e-3):
    rows = []
    selected_records = []

    for f in folds:
        train_df = df.iloc[f["train_idx"]].copy()
        valid_df = df.iloc[f["valid_idx"]].copy()

        rank_table = rank_features_by_train_correlation(train_df, feature_cols, target_col)

        for k in k_grid:
            selected = rank_table.head(k)["feature"].tolist()

            X_train, X_valid, _, _ = scale_train_valid(train_df, valid_df, selected)
            y_train = train_df[target_col].to_numpy()
            y_valid = valid_df[target_col].to_numpy()

            beta = fit_ridge(X_train, y_train, ridge_lambda)
            pred = predict_ridge(beta, X_valid)

            metrics = regression_metrics(y_valid, pred)

            rows.append({
                "split_type": f["split_type"],
                "fold": f["fold"],
                "k": k,
                "selected_features": "|".join(selected),
                **metrics
            })

            for feature in selected:
                selected_records.append({
                    "split_type": f["split_type"],
                    "fold": f["fold"],
                    "k": k,
                    "feature": feature
                })

    return pd.DataFrame(rows), pd.DataFrame(selected_records)


# Bad feature set includes the intentional leak.
cv_random_leaky, selected_random_leaky = run_cv_feature_selection(
    research,
    all_feature_cols,
    random_folds,
    config.k_grid,
    ridge_lambda=config.ridge_lambda
)

# Honest feature set excludes future-derived features.
cv_random_safe, selected_random_safe = run_cv_feature_selection(
    research,
    safe_feature_cols,
    random_folds,
    config.k_grid,
    ridge_lambda=config.ridge_lambda
)

cv_blocked_safe, selected_blocked_safe = run_cv_feature_selection(
    research,
    safe_feature_cols,
    blocked_folds,
    config.k_grid,
    ridge_lambda=config.ridge_lambda
)

cv_purged_safe, selected_purged_safe = run_cv_feature_selection(
    research,
    safe_feature_cols,
    purged_folds,
    config.k_grid,
    ridge_lambda=config.ridge_lambda
)

cv_results = pd.concat([
    cv_random_leaky.assign(experiment="random_kfold_with_leaky_feature"),
    cv_random_safe.assign(experiment="random_kfold_safe_features"),
    cv_blocked_safe.assign(experiment="blocked_kfold_safe_features"),
    cv_purged_safe.assign(experiment="purged_embargo_safe_features")
], ignore_index=True)

cv_results.head()

## 13. Naive CV can select leaky features

First, we show the bad experiment.

If the intentionally leaky feature is allowed, random CV will likely select it and report unrealistically strong validation performance.

This is not because the model is good.

It is because the feature is invalid.

In [ ]:
leaky_selection_counts = (
    selected_random_leaky
    .groupby(["k", "feature"])
    .size()
    .reset_index(name="selection_count")
    .sort_values(["k", "selection_count"], ascending=[True, False])
)

leaky_selection_counts[leaky_selection_counts["feature"].str.contains("LEAK", regex=False)].head()

In [ ]:
cv_summary = (
    cv_results
    .groupby(["experiment", "split_type", "k"])
    .agg(
        mean_mse=("mse", "mean"),
        mean_corr=("corr", "mean"),
        mean_directional_accuracy=("directional_accuracy", "mean"),
        mean_toy_ir_daily=("toy_strategy_ir_daily", "mean")
    )
    .reset_index()
)

cv_summary.sort_values("mean_mse").head(15)

In [ ]:
plt.figure(figsize=(12, 6))
for experiment, g in cv_summary.groupby("experiment"):
    plt.plot(g["k"], g["mean_corr"], marker="o", label=experiment)
plt.axhline(0, linestyle="--")
plt.title("CV Mean Correlation by Experiment")
plt.xlabel("Number of selected features")
plt.ylabel("Mean validation correlation")
plt.legend()
plt.show()

## 14. Purged CV versus naive CV

Now compare safe features only.

Random K-fold can still be over-optimistic because overlapping labels contaminate train/validation splits.

Purged + embargo CV is more conservative and closer to a live research setting.

In [ ]:
safe_cv_summary = cv_summary[cv_summary["experiment"] != "random_kfold_with_leaky_feature"].copy()

safe_cv_summary.sort_values(["experiment", "mean_mse"]).groupby("experiment").head(3)

In [ ]:
plt.figure(figsize=(12, 6))
for experiment, g in safe_cv_summary.groupby("experiment"):
    plt.plot(g["k"], g["mean_corr"], marker="o", label=experiment)
plt.axhline(0, linestyle="--")
plt.title("Safe Feature Set: CV Correlation by Split Method")
plt.xlabel("Number of selected features")
plt.ylabel("Mean validation correlation")
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
for experiment, g in safe_cv_summary.groupby("experiment"):
    plt.plot(g["k"], g["mean_mse"], marker="o", label=experiment)
plt.title("Safe Feature Set: CV MSE by Split Method")
plt.xlabel("Number of selected features")
plt.ylabel("Mean validation MSE")
plt.legend()
plt.show()

## 15. Select model complexity using purged CV

We choose $k$ using purged+embargo CV only.

This is the honest feature-selection protocol.

In [ ]:
purged_summary = safe_cv_summary[safe_cv_summary["experiment"] == "purged_embargo_safe_features"].copy()
best_k_row = purged_summary.sort_values("mean_mse").iloc[0]
best_k = int(best_k_row["k"])

best_k_row

## 16. Feature selection stability

A stable feature should be selected across folds.

We count how often each feature appears in the selected set for the best $k$.

In [ ]:
selected_purged_best = selected_purged_safe[selected_purged_safe["k"] == best_k].copy()

selection_stability = (
    selected_purged_best
    .groupby("feature")
    .agg(selection_count=("fold", "count"))
    .reset_index()
    .sort_values("selection_count", ascending=False)
)

selection_stability["selection_frequency"] = selection_stability["selection_count"] / config.n_splits

selection_stability

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(selection_stability["feature"][::-1], selection_stability["selection_frequency"][::-1])
plt.title(f"Purged CV Feature Selection Frequency, k={best_k}")
plt.xlabel("Selection frequency")
plt.ylabel("Feature")
plt.show()

## 17. Final feature set

We select final features using the full research set, with $k$ chosen by purged CV.

Then we train once on the full research set and evaluate on the untouched test set.

In [ ]:
final_rank_table = rank_features_by_train_correlation(research, safe_feature_cols, "forward_return_H")
final_features = final_rank_table.head(best_k)["feature"].tolist()

final_features

In [ ]:
X_research, X_test, final_mean, final_std = scale_train_valid(research, test, final_features)
y_research = research["forward_return_H"].to_numpy()
y_test = test["forward_return_H"].to_numpy()

final_beta = fit_ridge(X_research, y_research, config.ridge_lambda)
test_pred = predict_ridge(final_beta, X_test)

test_forecasts = test[["date", "t0", "t1", "forward_return_H", "forward_return_1d", "price"]].copy()
test_forecasts["forecast_purged_selected"] = test_pred

test_metrics = pd.Series(regression_metrics(y_test, test_pred))

test_metrics

## 18. Final test comparison with baselines

Baselines:

1. zero forecast;
2. research mean forecast;
3. momentum-only model;
4. bad leaky model for demonstration.

The leaky model is shown only to demonstrate how misleading leakage can be.

In [ ]:
# Zero and mean baselines.
test_forecasts["forecast_zero"] = 0.0
test_forecasts["forecast_research_mean"] = research["forward_return_H"].mean()

# Momentum-only baseline.
momentum_feature = ["momentum_21"]
X_res_mom, X_test_mom, _, _ = scale_train_valid(research, test, momentum_feature)
beta_mom = fit_ridge(X_res_mom, y_research, config.ridge_lambda)
test_forecasts["forecast_momentum_only"] = predict_ridge(beta_mom, X_test_mom)

# Bad leaky model demonstration: do not use in real research.
leaky_feature_set = ["LEAK_future_return_hint"]
X_res_leak, X_test_leak, _, _ = scale_train_valid(research, test, leaky_feature_set)
beta_leak = fit_ridge(X_res_leak, y_research, config.ridge_lambda)
test_forecasts["forecast_BAD_leaky_feature"] = predict_ridge(beta_leak, X_test_leak)

forecast_cols = [
    "forecast_zero",
    "forecast_research_mean",
    "forecast_momentum_only",
    "forecast_purged_selected",
    "forecast_BAD_leaky_feature"
]

test_comparison = pd.DataFrame([
    {"forecast": col, **regression_metrics(test_forecasts["forward_return_H"], test_forecasts[col])}
    for col in forecast_cols
]).sort_values("mse")

test_comparison

In [ ]:
plt.figure(figsize=(12, 6))
plt.barh(test_comparison["forecast"], test_comparison["corr"])
plt.axvline(0, linestyle="--")
plt.title("Untouched Test Correlation by Forecast")
plt.xlabel("Correlation")
plt.ylabel("Forecast")
plt.show()

plt.figure(figsize=(12, 6))
plot_sample = test_forecasts.iloc[:250]
plt.plot(plot_sample["date"], plot_sample["forward_return_H"], label="actual H-day return", alpha=0.65)
plt.plot(plot_sample["date"], plot_sample["forecast_purged_selected"], label="purged-selected forecast", alpha=0.85)
plt.axhline(0, linestyle="--")
plt.title("Final Test Forecast Sample")
plt.xlabel("Date")
plt.ylabel("Forward return")
plt.legend()
plt.show()

## 19. Toy strategy diagnostic

A forecast can be converted into a toy directional strategy:

$$
w_t=\operatorname{clip}\Big(\frac{\hat y_t}{2\sigma_{\hat y}},-1,1\Big)
$$

Strategy return proxy:

$$
R_t=w_t y_t - c|\Delta w_t|
$$

This is not a production backtest.

It is a diagnostic to see whether the forecast has any economic shape after turnover costs.

In [ ]:
def toy_strategy_from_forecast(df, forecast_col, target_col="forward_return_H", cost_per_turnover=0.0005):
    out = df[["date", forecast_col, target_col]].copy()
    pred = out[forecast_col].to_numpy()
    scale = np.std(pred, ddof=1)
    scale = scale if scale > 1e-12 else 1.0

    weight = np.clip(pred / (2 * scale), -1.0, 1.0)
    turnover = np.abs(np.diff(np.r_[0.0, weight]))
    gross = weight * out[target_col].to_numpy()
    cost = cost_per_turnover * turnover
    net = gross - cost

    out["weight"] = weight
    out["turnover"] = turnover
    out["gross_return_proxy"] = gross
    out["cost"] = cost
    out["net_return_proxy"] = net
    out["cum_net_return_proxy"] = np.cumsum(net)

    summary = {
        "forecast": forecast_col,
        "cost_per_turnover": cost_per_turnover,
        "mean_weight": float(np.mean(weight)),
        "mean_abs_weight": float(np.mean(np.abs(weight))),
        "mean_turnover": float(np.mean(turnover)),
        "total_gross_return_proxy": float(np.sum(gross)),
        "total_net_return_proxy": float(np.sum(net)),
        "mean_net_return_proxy": float(np.mean(net)),
        "vol_net_return_proxy": float(np.std(net, ddof=1)),
        "ir_proxy": float(np.mean(net) / np.std(net, ddof=1)) if np.std(net, ddof=1) > 0 else np.nan
    }

    return out, summary


strategy_frames = []
strategy_summaries = []

for col in forecast_cols:
    strat, summary = toy_strategy_from_forecast(test_forecasts, col, cost_per_turnover=0.0005)
    strat["forecast"] = col
    strategy_frames.append(strat)
    strategy_summaries.append(summary)

strategy_df = pd.concat(strategy_frames, ignore_index=True)
strategy_summary = pd.DataFrame(strategy_summaries).sort_values("ir_proxy", ascending=False)

strategy_summary

In [ ]:
plt.figure(figsize=(12, 6))
for forecast, g in strategy_df.groupby("forecast"):
    if forecast != "forecast_BAD_leaky_feature":
        plt.plot(g["date"], g["cum_net_return_proxy"], label=forecast)
plt.axhline(0, linestyle="--")
plt.title("Toy Strategy Diagnostic, Excluding Bad Leaky Feature")
plt.xlabel("Date")
plt.ylabel("Cumulative net return proxy")
plt.legend()
plt.show()

## 20. Cost sensitivity

Feature-selected models can look useful before costs.

We vary turnover cost to see whether the toy strategy survives.

In [ ]:
cost_grid = [0.0, 0.0001, 0.00025, 0.0005, 0.0010, 0.0020]
cost_rows = []

for cost in cost_grid:
    for col in ["forecast_momentum_only", "forecast_purged_selected"]:
        _, summary = toy_strategy_from_forecast(test_forecasts, col, cost_per_turnover=cost)
        cost_rows.append(summary)

cost_sensitivity = pd.DataFrame(cost_rows)

cost_sensitivity.head()

In [ ]:
plt.figure(figsize=(10, 6))
for forecast, g in cost_sensitivity.groupby("forecast"):
    plt.plot(g["cost_per_turnover"], g["ir_proxy"], marker="o", label=forecast)
plt.axhline(0, linestyle="--")
plt.title("Toy Strategy Cost Sensitivity")
plt.xlabel("Cost per turnover")
plt.ylabel("IR proxy")
plt.legend()
plt.show()

## 21. Feature correlation and redundancy in selected set

Even purged selection can select redundant features.

We inspect correlations among final selected features.

Highly correlated features may be candidates for orthogonalization or eviction in the factor-library process.

In [ ]:
selected_corr = research[final_features].corr()

selected_corr

In [ ]:
plt.figure(figsize=(7, 6))
plt.imshow(selected_corr.values, vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(final_features)), final_features, rotation=45, ha="right")
plt.yticks(range(len(final_features)), final_features)
plt.title("Correlation Among Final Selected Features")
plt.tight_layout()
plt.show()

## 22. Purged feature selection checklist

A robust time-series feature-selection process should check:

1. **Feature availability**  
   Does every feature exist at prediction time?

2. **Target horizon**  
   Is the forward label correctly shifted?

3. **Overlapping labels**  
   Do train and validation labels share future returns?

4. **Purging**  
   Are overlapping training events removed?

5. **Embargo**  
   Is a buffer applied after validation windows?

6. **Fold-internal selection**  
   Are features selected inside each fold only using training data?

7. **Train-only scaling**  
   Are scalers fit only on training fold?

8. **Untouched final test**  
   Is the final test set unused in selection?

9. **Leakage probes**  
   Would an intentionally leaky feature dominate if allowed?

10. **Costs and turnover**  
   Does selected model survive implementation costs?

11. **Stability**  
   Are selected features stable across folds?

12. **Redundancy**  
   Are selected features duplicates?

## 23. Saving outputs

The notebook saves:

1. synthetic event dataset;
2. feature metadata;
3. leakage audit;
4. fold diagnostics;
5. CV reports;
6. feature selection stability;
7. final selected features;
8. final test forecasts;
9. test comparison metrics;
10. toy strategy diagnostics;
11. cost sensitivity;
12. selected feature correlation;
13. manifest.

In [ ]:
output_dir = Path("data/processed/purged_feature_selection_for_time_series_v1")
output_dir.mkdir(parents=True, exist_ok=True)

events_path = output_dir / "synthetic_event_dataset.csv"
feature_metadata_path = output_dir / "feature_metadata.csv"
leakage_audit_path = output_dir / "leakage_audit.csv"
fold_diagnostics_path = output_dir / "fold_diagnostics.csv"
cv_results_path = output_dir / "cv_results.csv"
cv_summary_path = output_dir / "cv_summary.csv"
selection_stability_path = output_dir / "selection_stability.csv"
final_rank_path = output_dir / "final_feature_rank.csv"
final_features_path = output_dir / "final_selected_features.json"
test_forecasts_path = output_dir / "test_forecasts.csv"
test_comparison_path = output_dir / "test_comparison.csv"
strategy_summary_path = output_dir / "strategy_summary.csv"
cost_sensitivity_path = output_dir / "cost_sensitivity.csv"
selected_corr_path = output_dir / "selected_feature_correlation.csv"
split_metadata_path = output_dir / "split_metadata.json"
manifest_path = output_dir / "manifest.json"

events.to_csv(events_path, index=False)
feature_metadata.to_csv(feature_metadata_path, index=False)
leakage_audit.to_csv(leakage_audit_path, index=False)
fold_diagnostics.to_csv(fold_diagnostics_path, index=False)
cv_results.to_csv(cv_results_path, index=False)
cv_summary.to_csv(cv_summary_path, index=False)
selection_stability.to_csv(selection_stability_path, index=False)
final_rank_table.to_csv(final_rank_path, index=False)
Path(final_features_path).write_text(json.dumps({"best_k": best_k, "final_features": final_features}, indent=2))
test_forecasts.to_csv(test_forecasts_path, index=False)
test_comparison.to_csv(test_comparison_path, index=False)
strategy_summary.to_csv(strategy_summary_path, index=False)
cost_sensitivity.to_csv(cost_sensitivity_path, index=False)
selected_corr.to_csv(selected_corr_path)
Path(split_metadata_path).write_text(json.dumps(split_metadata, indent=2, default=str))

manifest = {
    "dataset_name": "purged_feature_selection_for_time_series_outputs",
    "schema_version": "purged_feature_selection_for_time_series_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "all_feature_cols": all_feature_cols,
    "safe_feature_cols": safe_feature_cols,
    "leaky_features_removed": leaky_features,
    "best_k_purged_cv": best_k,
    "final_features": final_features,
    "test_metrics_purged_selected": test_metrics.to_dict(),
    "best_test_forecast_by_mse": test_comparison.sort_values("mse").iloc[0].to_dict(),
    "notes": [
        "Synthetic event dataset uses overlapping forward-return labels.",
        "Intentional leaky feature is included to demonstrate feature availability audits.",
        "Random K-fold, blocked K-fold, and purged embargo K-fold are compared.",
        "Feature selection is performed inside each fold only using training data.",
        "Final test set is untouched until after purged CV selects model complexity.",
        "Toy strategy is a diagnostic only, not a production trading backtest."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

events_path, cv_results_path, test_forecasts_path, test_comparison_path, manifest_path

## 24. Limitations

### 24.1 Synthetic data

This notebook uses synthetic data.

Real data requires careful handling of vendor timestamps, publication delays, revisions, missing observations, corporate actions, contract rolls, and market calendars.

### 24.2 Simple ridge model

The model is intentionally simple.

The point is validation design, not model complexity.

### 24.3 Purging does not fix all leakage

Purging handles overlapping labels.

It does not fix:

- future-derived features;
- incorrect data timestamps;
- survivorship bias;
- target leakage through preprocessing;
- universe selection leakage.

### 24.4 Embargo length is a governance choice

The correct embargo depends on label horizon, feature window, market microstructure, and serial dependence.

### 24.5 Feature selection is filter-based

Feature ranking by train correlation is simple.

Production workflows may use nested model-based selection, stability selection, or regularisation.

### 24.6 Overlapping labels need adjusted inference

Even after purging, final performance metrics may have serial dependence if labels overlap.

### 24.7 Toy strategy is simplified

The strategy diagnostic ignores realistic execution, slippage, capacity, and risk constraints.

## 25. Research frontier and extensions

Important extensions include:

1. **Combinatorial purged cross-validation**  
   More robust evaluation across many train/test path combinations.

2. **Purged walk-forward validation**  
   Live-like rolling retraining with embargo.

3. **Nested purged feature selection**  
   Feature selection and hyperparameter tuning inside nested folds.

4. **Sequential bootstrap**  
   Resample events while accounting for label overlap.

5. **Purged model stacking**  
   Prevent meta-model leakage from base learner predictions.

6. **Point-in-time feature store**  
   Enforce availability timestamps for every feature.

7. **Multiple-testing correction**  
   Control false discoveries across many feature trials.

8. **Purged SHAP/importance analysis**  
   Compute feature importance within leakage-safe folds.

9. **Cross-sectional purged validation**  
   Combine time purging with asset-level grouping.

10. **Chinese futures feature selection**  
   Apply purging to rolling contracts, night sessions, L1 tick features, and overlapping forward-return labels.

## 26. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_16_tree_models_for_return_prediction**  
   Apply purged feature selection to nonlinear models.

2. **03_17_feature_importance_and_shap_for_alpha**  
   Compute feature importance without leakage.

3. **05_05_walk_forward_model_validation**  
   General model validation infrastructure.

4. **05_06_combinatorial_purged_cross_validation**  
   Formal CPCV implementation.

5. **07_01_multi_asset_cta_research_pipeline**  
   Use purged feature selection in the full futures research pipeline.

## 27. Summary

This notebook implemented purged feature selection for time-series ML.

It showed how to:

1. create overlapping forward-return event labels;
2. identify feature look-ahead leakage;
3. compare random, blocked, and purged CV;
4. apply embargo after validation windows;
5. select features inside each fold;
6. compare leaky and honest experiments;
7. choose model complexity using purged CV;
8. measure feature selection stability;
9. train a final model on the research set;
10. evaluate on an untouched test set;
11. compare baselines and leaky-feature failure modes;
12. run toy strategy and cost diagnostics;
13. save reports and manifests.

The key computational takeaway:

> Feature selection must be inside a leakage-safe validation loop, not performed once on the full dataset.

The key financial takeaway:

> Many apparent alpha features are validation artifacts unless event overlap, embargo, and feature availability are handled explicitly.

## 28. Further reading

- López de Prado, *Advances in Financial Machine Learning.*
- Literature on purged K-fold cross-validation and combinatorial purged cross-validation.
- Literature on leakage in financial machine learning, event-based labelling, and sequential bootstrapping.
- Research on feature stores, point-in-time data, and model validation governance.